# CatBoost RMSE Optimization with 100+ Engineered Features

In [ ]:

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

SEED = 42
FOLDS = 5

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
TARGET = "Calories_Burned"


In [ ]:

def feature_engineering(df):
    df = df.copy()
    
    # Height in inches
    df["Height_Inches"] = df["Height(Feet)"] * 12 + df["Height(Remainder_Inches)"]
    
    # BMI
    df["BMI"] = 703 * df["Weight(lb)"] / (df["Height_Inches"] ** 2 + 1e-6)
    
    # BSA (Body Surface Area)
    df["BSA"] = 0.007184 * (df["Weight(lb)"] ** 0.425) * (df["Height_Inches"] ** 0.725)
    
    # Heart related
    df["HRxDur"] = df["BPM"] * df["Exercise_Duration"]
    df["TempHR"] = df["Body_Temperature(F)"] * df["BPM"]
    df["TempDur"] = df["Body_Temperature(F)"] * df["Exercise_Duration"]
    
    # Age interactions
    df["AgeHR"] = df["Age"] * df["BPM"]
    df["AgeDur"] = df["Age"] * df["Exercise_Duration"]
    
    # Ratios
    df["HR_per_weight"] = df["BPM"] / (df["Weight(lb)"] + 1e-6)
    df["Dur_per_weight"] = df["Exercise_Duration"] / (df["Weight(lb)"] + 1e-6)
    
    # Polynomial features
    num_cols = ["Exercise_Duration", "Body_Temperature(F)", "BPM", "Weight(lb)", "Age", "Height_Inches"]
    
    for col in num_cols:
        df[f"{col}_sq"] = df[col] ** 2
        df[f"{col}_cube"] = df[col] ** 3
    
    # Cross interactions (generate many features)
    for i, c1 in enumerate(num_cols):
        for c2 in num_cols[i+1:]:
            df[f"{c1}_x_{c2}"] = df[c1] * df[c2]
            df[f"{c1}_div_{c2}"] = df[c1] / (df[c2] + 1e-6)
            df[f"{c2}_div_{c1}"] = df[c2] / (df[c1] + 1e-6)
    
    return df

train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

cat_cols = ["Gender", "Weight_Status"]
features = [c for c in train_fe.columns if c not in ["ID", TARGET]]

X = train_fe[features]
y = train_fe[TARGET]
X_test = test_fe[features]


In [ ]:

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

oof = np.zeros(len(X))
preds = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y)):
    print(f"Fold {fold}")
    
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    
    train_pool = Pool(X_tr, y_tr, cat_features=cat_cols)
    valid_pool = Pool(X_va, y_va, cat_features=cat_cols)
    
    model = CatBoostRegressor(
        iterations=8000,
        depth=8,
        learning_rate=0.03,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=SEED,
        verbose=500,
        early_stopping_rounds=300
    )
    
    model.fit(train_pool, eval_set=valid_pool)
    
    oof[va_idx] = model.predict(X_va)
    preds += model.predict(X_test) / FOLDS

rmse = mean_squared_error(y, oof, squared=False)
print("Overall RMSE:", rmse)


In [ ]:

submission = pd.DataFrame({
    "ID": test["ID"],
    "Calories_Burned": preds
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
